In [ ]:
# https://docs.pytorch.org/tutorials/intermediate/FSDP_tutorial.html#fsdp1-to-fsdp2-migration-guide

<img src="https://r2cdn.perplexity.ai/pplx-full-logo-primary-dark%402x.png" style="height:64px;margin-right:32px"/>

# can you do the whole fsdp pipeline with 4 gpus for this simple example? Global weight: [w0,w1,w2,w3][w0,w1,w2,w3]

Consider 4 GPUs (ranks 0,1,2,3) and a *single* FSDP unit with a “global weight” vector of length 4:

$$
W = [w_0, w_1, w_2, w_3]
$$

You wrote it twice, so treat this as one 4‑element parameter, not two layers. The important part is how this one 4‑element thing is handled across 4 GPUs.[^1][^2]

***

## 0. Sharding layout (between steps)

World size = 4. FSDP shards this weight into 4 pieces:

- Rank 0 stores: $[w_0]$
- Rank 1 stores: $[w_1]$
- Rank 2 stores: $[w_2]$
- Rank 3 stores: $[w_3]$

Between forward/backward passes, each rank only has its one element and its optimizer state for that element.[^3][^2][^1]

***

## 1. Forward pass (for this FSDP unit)

When code enters this FSDP‑wrapped module:

1. **All‑gather params**

Each rank all‑gathers the 4 shards:
    - Before:
        - Rank 0: [w0]
        - Rank 1: [w1]
        - Rank 2: [w2]
        - Rank 3: [w3]
    - After all‑gather, each rank temporarily reconstructs full weight:
        - Rank 0: [w0, w1, w2, w3]
        - Rank 1: [w0, w1, w2, w3]
        - Rank 2: [w0, w1, w2, w3]
        - Rank 3: [w0, w1, w2, w3]

This is the “run all_gather to recover full parameter” step.[^2][^1]
2. **Compute local forward**

Each rank uses the full `[w0,w1,w2,w3]` with its *own* data shard to compute outputs and store activations for autograd. (This part is just normal forward.)
3. **Re‑shard / discard full params**

After this unit’s forward is done, FSDP discards the full `[w0,w1,w2,w3]` on each rank and keeps only the local shard:
    - Rank 0 keeps [w0]
    - Rank 1 keeps [w1]
    - Rank 2 keeps [w2]
    - Rank 3 keeps [w3]

So forward is “gather full param → compute → go back to shards”.[^1][^2]

***

## 2. Backward pass (for this FSDP unit)

When autograd reaches this FSDP unit during backward:

1. **All‑gather params again (if needed)**

To compute correct gradients, FSDP makes full `[w0,w1,w2,w3]` visible again (same all‑gather).[^2][^1]
2. **Compute local gradients**

Each rank’s autograd now computes gradients with respect to the full weight vector, based on its local data. So each rank *temporarily* has a **full gradient vector**:
    - Rank 0: $g^{(0)} = [g^{(0)}_0, g^{(0)}_1, g^{(0)}_2, g^{(0)}_3]$
    - Rank 1: $g^{(1)} = [g^{(1)}_0, g^{(1)}_1, g^{(1)}_2, g^{(1)}_3]$
    - Rank 2: $g^{(2)} = [g^{(2)}_0, g^{(2)}_1, g^{(2)}_2, g^{(2)}_3]$
    - Rank 3: $g^{(3)} = [g^{(3)}_0, g^{(3)}_1, g^{(3)}_2, g^{(3)}_3]$

True global gradient is elementwise sum over ranks:

$$
g = g^{(0)} + g^{(1)} + g^{(2)} + g^{(3)}
  = [g_0, g_1, g_2, g_3].
$$
3. **Reduce‑scatter: “sum then split”**

Now FSDP calls **reduce‑scatter** over these 4‑element gradients.[^4][^1][^2]

Conceptually it does:
    - **Reduce (sum across ranks):**

Compute global `g = [g0, g1, g2, g3]` as above.
    - **Scatter (give each rank 1 element):**
        - Rank 0 keeps `g0`
        - Rank 1 keeps `g1`
        - Rank 2 keeps `g2`
        - Rank 3 keeps `g3`

After reduce‑scatter, each rank’s gradient buffer is only length 1:
    - Rank 0: `[g0]`
    - Rank 1: `[g1]`
    - Rank 2: `[g2]`
    - Rank 3: `[g3]`

Each is **already summed over all 4 ranks**, but only for that rank’s own weight element.[^3][^1][^2]
4. **Discard full params again**

FSDP frees the full `[w0,w1,w2,w3]` view; only shards and sharded grads remain.[^1][^2]

***

## 3. Optimizer step (still same iteration)

Now each rank has:

- Param shard:
    - Rank 0: `[w0]`
    - Rank 1: `[w1]`
    - Rank 2: `[w2]`
    - Rank 3: `[w3]`
- Gradient shard (already reduced):
    - Rank 0: `[g0]`
    - Rank 1: `[g1]`
    - Rank 2: `[g2]`
    - Rank 3: `[g3]`

Optimizer runs **locally** per rank:

- Rank 0 updates `w0` using `g0`.
- Rank 1 updates `w1` using `g1`.
- Rank 2 updates `w2` using `g2`.
- Rank 3 updates `w3` using `g3`.[^2][^1]

After this step, conceptually you have updated global weight:

$$
W' = [w'_0, w'_1, w'_2, w'_3]
$$

but it is still stored in sharded form across ranks exactly like before.

***

## 4. Next iteration

Next training iteration, when you enter this FSDP unit again:

- Each rank starts from its updated shard:
    - Rank 0: `[w0']`
    - Rank 1: `[w1']`
    - Rank 2: `[w2']`
    - Rank 3: `[w3']`
- **Forward**:
    - All‑gather → each rank temporarily sees `[w0',w1',w2',w3']`.
    - Compute; then re‑shard.
- **Backward**:
    - All‑gather params (if needed).
    - Compute full local grads.
    - Reduce‑scatter → each rank keeps 1 gradient element (`g0', g1', g2', g3'`).
    - Optimizer updates its shard.[^1][^2]

This repeats every step.

***

So with your `[w0,w1,w2,w3]` and 4 GPUs, the “whole FSDP pipeline” is:

- Between steps: each GPU has 1 param number + its optimizer state.
- Forward: all‑gather to full vector on each GPU → compute → drop back to 1 number.
- Backward: all‑gather params → compute full grads → reduce‑scatter to 1 grad number per GPU → update that 1 param.
<span style="display:none">[^10][^5][^6][^7][^8][^9]</span>

<div align="center">⁂</div>

[^1]: https://pytorch-cn.com/tutorials/intermediate/FSDP_tutorial.html

[^2]: https://docs.pytorch.org/tutorials/intermediate/FSDP_tutorial.html

[^3]: https://engineering.fb.com/2021/07/15/open-source/fsdp/attachment/fsdp-graph-2a/

[^4]: https://lightning-thunder.readthedocs.io/en/stable/notebooks/dev_tutorials/fsdp_tutorial.html

[^5]: https://discuss.pytorch.org/t/how-does-fsdp-algorithm-work/173277

[^6]: https://sumanthrh.com/post/distributed-and-efficient-finetuning/

[^7]: https://www.linkedin.com/pulse/rl-system-deep-dive-fsdp-training-backend-chenyang-zhao-jjeoe

[^8]: https://docs.nvidia.com/megatron-core/developer-guide/latest/api-guide/custom_fsdp.html

[^9]: https://github.com/pytorch/pytorch/issues/120879

[^10]: https://docs.pytorch.org/tutorials/_sources/intermediate/FSDP_tutorial.rst.txt



<img src="https://r2cdn.perplexity.ai/pplx-full-logo-primary-dark%402x.png" style="height:64px;margin-right:32px"/>

# so when you dont shard params and grads, only optimizer, how does this work? like in zero-1

In ZeRO‑1 (or “optimizer‑only sharding”), **yes: params and grads are still replicated; only the optimizer state is partitioned across ranks.**[^1][^2]

## What is sharded vs replicated in ZeRO‑1

For data‑parallel degree $N$:

- **Parameters:**
    - Fully replicated on every GPU, just like vanilla DDP.
    - Each rank holds the entire model weights.[^3][^2]
- **Gradients:**
    - Also fully present (logically) on every GPU after backward (via all‑reduce), same as DDP.[^2][^3]
- **Optimizer state (ZeRO‑1):**
    - **Sharded/partitioned** across GPUs: each rank stores only $1/N$ of the optimizer state tensors (Adam’s $m, v$, FP32 master weights, etc.).[^4][^1][^2]
    - On its shard, a rank updates the corresponding slice of parameters using the *full* grads it has locally, then those updated param slices are broadcast/all‑gathered so everyone’s full model stays in sync.[^5][^6]

So for your list:

- **optimizer** → sharded (ZeRO‑1)
- **optimizer + grads** → ZeRO‑2 (optimizer + gradients sharded, params still replicated).[^1][^2]
- **optimizer + grads + weights** → ZeRO‑3 (all three sharded, like “ZeRO‑style FSDP”).[^3][^2][^1]
<span style="display:none">[^10][^11][^12][^13][^14][^15][^16][^17][^18][^19][^20][^7][^8][^9]</span>

<div align="center">⁂</div>

[^1]: https://www.byteplus.com/en/topic/497900

[^2]: https://deepspeed.readthedocs.io/en/latest/zero3.html

[^3]: https://apxml.com/courses/distributed-training-pytorch-fsdp/chapter-1-limits-data-parallelism-zero-fundamentals/zero-stages-sharding-strategies

[^4]: https://www.type.sh/index.php/2024/03/10/zero-zero-redundancy-optimizer-explained/

[^5]: https://docs.pytorch.org/tutorials/recipes/zero_redundancy_optimizer.html

[^6]: https://dudeperf3ct.github.io/posts/ultrascale_zero_deepspeed/

[^7]: https://awsdocs-neuron.readthedocs-hosted.com/en/latest/frameworks/torch/torch-neuronx/tutorials/training/zero1_gpt2.html

[^8]: https://deepspeed.readthedocs.io/en/stable/zero3.html

[^9]: https://engineering.fb.com/2021/07/15/open-source/fsdp/

[^10]: https://haroldbenoit.com/notes/ml/engineering/training/parallelism/zero-redundancy-optimizer

[^11]: https://colossalai.org/docs/features/zero_with_chunk/

[^12]: https://dev.to/lewis_won/data-parallelism-4g3m

[^13]: https://haroldbenoit.com/notes/ML/Engineering/Training/Parallelism/Zero-Redundancy-Optimizer

[^14]: https://oslo.eleuther.ai/TUTORIALS/zero_redundancy_optimizer.html

[^15]: https://www.microsoft.com/en-us/research/blog/zero-deepspeed-new-system-optimizations-enable-training-models-with-over-100-billion-parameters/

[^16]: https://www.deepspeed.ai/tutorials/zero/

[^17]: https://docs.oneflow.org/en/master/cookies/zero.html

[^18]: https://apxml.com/courses/how-to-build-a-large-language-model/chapter-16-implementing-distributed-training-frameworks/using-deepspeed-zero-optimizations

[^19]: https://huggingface.co/docs/accelerate/v0.10.0/en/deepspeed

[^20]: http://oslo.eleuther.ai/TUTORIALS/zero_redundancy_optimizer.html



In [ ]:
# Next: TP: https://docs.pytorch.org/tutorials/intermediate/TP_tutorial.html

In [1]:
## TP

## Why TP + FSDP is helpful

(1) Communication latency at huge GPU counts: When FSDP runs across very large world sizes (e.g., 128/256+ GPUs), collectives like all_gather get slow because latency grows with the length of the communication ring, so GPUs spend more time waiting than computing; adding TP/SP lets you run FSDP mostly inter-node (treating each 8‑GPU node as a unit), shrinking the FSDP world size ~8× and cutting that latency cost accordingly.

(2) Global batch size ceiling: With pure data parallelism, scaling GPU count tends to force global batch size up (or local batch size down to tiny values), which can hit convergence and memory limits; TP/SP is the practical way to keep scaling model size + GPU count without requiring an impractically large global batch size.

(3) Better matmul shapes at small local batch: When local batch per GPU gets small, TP/SP can reshape the matrix multiplications into dimensions that the GPU kernels handle more efficiently, improving achieved FLOPS for some models.

Summary: In practice, we usually apply Tensor Parallel within each host, and apply Fully Sharded Data Parallel across the hosts.

## Example of FSDP with TP

Here is a summary of the FSDP + TP mechanism, illustrated with the **2-Node (16 GPUs total)** example.

### 1. The Setup: 2 Nodes, 8 GPUs Each
- **Total GPUs:** 16 (Nodes A & B).
- **Configuration:** **Tensor Parallel (TP) = 8** (Intra-Node), **FSDP = 2** (Inter-Node).
- **Communication Hardware:**
    - **Inside Node:** Ultra-fast NVLink (for TP).
    - **Between Nodes:** Slower Ethernet/InfiniBand (for FSDP).
- **Parallel Lanes:** Each GPU has its own dedicated network card (NIC), allowing all 8 GPUs to talk to their remote counterparts simultaneously.

### 2. How the Model is Split (The "Shard of a Shard")
Instead of sharding the entire model across all 16 GPUs, we split it in two steps:
1.  **Vertical Cut (TP):** The weight matrix (e.g., 800x800) is sliced into 8 vertical strips (800x100).
    - *GPU 0 on Node A* and *GPU 0 on Node B* are both assigned **Strip #1**.
2.  **Horizontal Cut (FSDP):** To avoid redundancy, FSDP splits that strip in half.
    - **Node A, GPU 0:** Stores Top Half (Rows 0–400) of Strip 1.
    - **Node B, GPU 0:** Stores Bottom Half (Rows 401–800) of Strip 1.
    - *Result:* Each GPU holds 1/16th of the total model, just like pure FSDP, but organized differently.[2][3]

### 3. The Execution Flow (Step-by-Step)
**A. The Inter-Node Handshake (FSDP All-Gather)**
- **Action:** Node A and Node B need to rebuild their assigned TP Strips.
- **Process:** GPU 0 (Node A) swaps halves with GPU 0 (Node B). All 8 pairs do this simultaneously on parallel lanes.
- **Latency:** **Low.** Instead of a 16-step ring (Pure FSDP), it’s effectively a 1-step direct swap between 2 neighbors.[1]

**B. The Local Compute (TP Processing)**
- **Action:** Now that GPU 0 (Node A) has the full Strip #1, it computes its slice of the math.
- **Process:** It multiplies its input batch by Strip #1.

**C. The Intra-Node Sync (TP All-Reduce)**
- **Action:** To get the final layer output, the 8 GPUs on Node A must sum their partial results.
- **Process:** They talk over **NVLink** (ultra-fast). This happens frequently (every layer) but is nearly instant.[2][3]

### 4. Why This Wins
- **Latency Reduction:** Replaces a global 16-way traffic jam with parallel 2-way phone calls.
- **Data Parallel Efficiency:** Allows you to use 16 GPUs but only process **2 images** (Batch Size = 2) instead of forcing a batch size of 16.
- **Memory:** Keeps the same memory efficiency as pure FSDP (1/N memory usage) but drastically improves communication speed.[3][1]

[1](https://lightning.ai/docs/pytorch/stable/advanced/model_parallel/tp_fsdp.html)
[2](https://blog.ezyang.com/2025/08/the-parallelism-mesh-zoo/)
[3](https://www.perplexity.ai/search/d7848bd6-0ce8-484e-96c4-e6449db27b10)
[4](https://www.perplexity.ai/search/15979427-8171-4eff-98c4-c7dd96a35b8f)

## Sequence Parallelism (SP) Summary

### The Problem with Pure Tensor Parallel (TP)
While TP efficiently splits the heavy Matrix Multiplications (MatMul) across GPUs, it redundantly copies the activations for "light" operations (LayerNorm, Dropout) on every single GPU.
- **Waste:** With TP=8, you store 8 identical copies of the activations.[1][2]

### The Solution (Sequence Parallel)
Instead of replicating the full sequence on every GPU, SP **splits the sequence (tokens)** across the TP group during the light operations.
- **Example:** For an 8-token sentence on 8 GPUs, GPU 0 processes only Token 1, GPU 1 processes only Token 2, etc.
- **Benefit:** Reduces activation memory usage by a factor of $N$ (TP size).[3][1]

### How LayerNorm Works in SP
- **Clarification:** LayerNorm normalizes across the **Hidden Dimension (Features)**, not the Sequence Length.
- **Execution:** Since GPU 0 holds *all* the features (e.g., 4096 columns) for its specific token (e.g., "The"), it can compute the mean and variance independently without talking to other GPUs. No communication is needed for LayerNorm in SP.[4][1]

### The Workflow (Transitioning States)
1.  **Light Ops (LayerNorm/Dropout):** Data is split by **Sequence**. (Each GPU holds 1/8th of the tokens, full hidden dim).
2.  **Transition 1 (All-Gather):** Before MatMul, GPUs gather tokens so they can switch to TP format.
3.  **Heavy Ops (MatMul):** Data is split by **Hidden Dimension**. (Each GPU holds all tokens, 1/8th of the hidden dim).
4.  **Transition 2 (Reduce-Scatter):** After MatMul, GPUs sum results and scatter them back to "Sequence Split" mode.[2]

[1](https://insujang.github.io/2024-01-11/tensor-parallelism-and-sequence-parallelism-detailed-analysis/)
[2](https://haroldbenoit.com/notes/ml/engineering/training/parallelism/sequence-parallelism)
[3](https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/features/parallelisms.html)
[4](https://triton-lang.org/main/getting-started/tutorials/05-layer-norm.html)

In [2]:
# Next: https://docs.pytorch.org/tutorials/intermediate/pipelining_tutorial.html